In [ ]:
!curl -Lo memfix.zip https://github.com/nolanaatama/microsoftexcel/raw/main/memfix.zip
!unzip -o /content/memfix.zip
!apt -y update -qq
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

# [FIX] Colab now runs Python 3.13, and this WebUI (July 2023) needs Python 3.10 with its original
# package versions. These lines build that environment in /content/sdenv. Nothing else about the WebUI changes.
open('/content/constraints.txt', 'w').write('torch==2.0.1\ntorchvision==0.15.2\nxformers==0.0.20\nnumpy<2\n')
%env PIP_CONSTRAINT=/content/constraints.txt
!pip install -q uv
!uv python install 3.10
![ -x /content/sdenv/bin/python ] || uv venv -q --seed --python 3.10 /content/sdenv
!/content/sdenv/bin/python -m pip install -q pip==24.0 setuptools==69.5.1 wheel
!/content/sdenv/bin/python -m pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 --index-url https://download.pytorch.org/whl/cu118
!/content/sdenv/bin/python -m pip install -q --no-deps xformers==0.0.20
!/content/sdenv/bin/python -m pip install -q --no-build-isolation https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip

!curl -Lo microsoftexcel.zip https://huggingface.co/nolanaatama/colab/resolve/main/microsoftexcel151.zip
!unzip -o -q /content/microsoftexcel.zip

# [FIX] Keep newer package releases from breaking the WebUI. MicrosoftExcel's own pinned versions
# (requirements_versions.txt) always win; these only cover packages that file leaves unpinned.
import re, json, urllib.request
_reqs = open('/content/microsoftexcel/requirements_versions.txt').read()
_name = lambda l: re.split(r'[<>=!~ ;\[]', l.strip())[0].lower().replace('_', '-')
_pinned = {_name(l) for l in _reqs.splitlines() if l.strip() and not l.lstrip().startswith('#')}
_extra = ['numpy<2', 'Pillow<10', 'pydantic<2', 'httpx==0.24.1', 'websockets==11.0.3', 'uvicorn==0.23.2',
          'huggingface-hub<0.26', 'opencv-python<4.12', 'opencv-python-headless<4.12', 'opencv-contrib-python<4.12',
          'protobuf<4', 'mediapipe==0.10.9', 'diffusers<0.21', 'ultralytics<8.1']
_g = re.search(r'^gradio==([\w.]+)', _reqs, re.M)
if _g:  # gradio 3.x accepts any newer gradio_client, but gradio_client 1.0+ breaks it: use the version it was released with
    for _r in json.load(urllib.request.urlopen(f'https://pypi.org/pypi/gradio/{_g.group(1)}/json'))['info']['requires_dist'] or []:
        _m = re.match(r'gradio[-_]client\s*(?:>=|==)\s*([\w.]+)', _r, re.I)
        if _m: _extra.append(f'gradio_client=={_m.group(1)}')
open('/content/constraints.txt', 'w').write('\n'.join(l for l in ['torch==2.0.1', 'torchvision==0.15.2', 'xformers==0.0.20'] + _extra if _name(l) not in _pinned) + '\n')
print('MicrosoftExcel pins:', ', '.join(l.strip() for l in _reqs.splitlines() if '==' in l))
print('Extra pins:', open('/content/constraints.txt').read().replace('\n', ', '))

# (OPTIONAL) Google Drive: keep your models between sessions
# Remove '#' from the lines below. Models then live in Google Drive > MyDrive > microsoftexcel.
# Once a model is saved there, put '#' back in front of its !curl line so it isn't downloaded again.
#from google.colab import drive; drive.mount('/content/drive')
#for d in ['models/Stable-diffusion', 'models/Lora', 'models/VAE', 'models/ESRGAN', 'embeddings']:
#    src = f'/content/drive/MyDrive/microsoftexcel/{d}'; dst = f'/content/microsoftexcel/{d}'
#    !mkdir -p "{src}" && cp -rn "{dst}/." "{src}/" 2>/dev/null; rm -rf "{dst}" && ln -s "{src}" "{dst}"

# Extensions Section
!git clone https://github.com/nolanaatama/microsoftexcel-tunnels /content/microsoftexcel/extensions/microsoftexcel-tunnels
!git clone https://github.com/nolanaatama/microsoftexcel-controlnet /content/microsoftexcel/extensions/microsoftexcel-controlnet
!git clone https://github.com/fkunn1326/openpose-editor /content/microsoftexcel/extensions/openpose-editor
!git clone https://github.com/nolanaatama/microsoftexcel-3d-open-pose-editor /content/microsoftexcel/extensions/microsoftexcel-3d-open-pose-editor
!git clone https://github.com/nolanaatama/a1111-microsoftexcel-tagcomplete /content/microsoftexcel/extensions/a1111-microsoftexcel-tagcomplete
!git clone https://github.com/nolanaatama/a1111-microsoftexcel-locon /content/microsoftexcel/extensions/a1111-microsoftexcel-locon
# Additional Extensions
# !After Detailer
# [FIX] pinned to v23.8.1, the last ADetailer release for this WebUI version (later ones need WebUI 1.6+)
#!git clone -b v23.8.1 https://github.com/Bing-su/adetailer /content/microsoftexcel/extensions/adetailer
#!git clone https://huggingface.co/Bingsu/adetailer /content/microsoftexcel/models/adetailer
# Deforum
#!git clone https://github.com/nolanaatama/microsoftexcel-deforum /content/microsoftexcel/extensions/microsoftexcel-deforum
# Mov2mov
#!git clone https://github.com/nolanaatama/microsoftexcel-mov2mov /content/microsoftexcel/extensions/microsoftexcel-mov2mov
# roop  (note: Colab's terms forbid creating deepfakes on every runtime, and roop's pinned opencv conflicts with ControlNet)
#!git clone https://github.com/nolanaatama/microsoftexcel-roop /content/microsoftexcel/extensions/microsoftexcel-roop
# SuperMerger
#!git clone https://github.com/nolanaatama/microsoftexcel-supermerger /content/microsoftexcel/extensions/microsoftexcel-supermerger
# TemporalKit
#!git clone https://github.com/CiaraStrawberry/TemporalKit /content/microsoftexcel/extensions/TemporalKit
# Ultimate SD Upscale
#!git clone https://github.com/Coyote-A/ultimate-upscale-for-automatic1111 /content/microsoftexcel/extensions/ultimate-upscale-for-automatic1111

!mkdir -p /content/microsoftexcel/models/ESRGAN
!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-images-browser.zip https://huggingface.co/nolanaatama/colab/resolve/main/microsoftexcel-images-browser.zip
!curl -Lo /content/microsoftexcel/embeddings/embeddings.zip https://huggingface.co/nolanaatama/colab/resolve/main/embeddings.zip
!curl -Lo /content/microsoftexcel/models/ESRGAN/upscalers.zip https://huggingface.co/nolanaatama/colab/resolve/main/upscalers.zip
%cd /content/microsoftexcel/extensions
!unzip -o /content/microsoftexcel/extensions/microsoftexcel-images-browser.zip
%cd /content/microsoftexcel/embeddings
!unzip -o /content/microsoftexcel/embeddings/embeddings.zip
%cd /content/microsoftexcel/models/ESRGAN
!unzip -o /content/microsoftexcel/models/ESRGAN/upscalers.zip
!rm upscalers.zip
%cd /content

# Model Code
!curl -Lo /content/microsoftexcel/models/Stable-diffusion/epicrealism.safetensors https://huggingface.co/nolanaatama/pcrlsm/resolve/main/pcrlsm.safetensors

# ControlNet
# Remove '#' from the beginning of the line(s) below to download the selected ControlNet model(s)
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11e_sd15_ip2p.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11e_sd15_ip2p_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11e_sd15_shuffle.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11e_sd15_shuffle_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_canny.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_canny_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11f1p_sd15_depth.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_depth_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_inpaint.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_inpaint_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_lineart.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_lineart_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_mlsd.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_mlsd_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_normalbae.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_normalbae_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_openpose.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_openpose_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_scribble.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_scribble_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_seg.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_seg_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15_softedge.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15_softedge_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11p_sd15s2_lineart_anime.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11p_sd15s2_lineart_anime_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/control_v11f1e_sd15_tile.safetensors https://huggingface.co/nolanaatama/models/resolve/main/control_v11f1e_sd15_tile_fp16.safetensors
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_canny_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_canny_sd14v1.pth
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_color_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_color_sd14v1.pth
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_depth_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_depth_sd14v1.pth
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_keypose_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_keypose_sd14v1.pth
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_openpose_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_openpose_sd14v1.pth
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_seg_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_seg_sd14v1.pth
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_sketch_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_sketch_sd14v1.pth
#!curl -Lo /content/microsoftexcel/extensions/microsoftexcel-controlnet/models/t2iadapter_style_sd14v1.pth https://huggingface.co/nolanaatama/models/resolve/main/t2iadapter_style_sd14v1.pth
!rm microsoftexcel.zip
%cd /content/microsoftexcel

# Web UI tunnel
# [FIX] GitHub's Stability-AI/stablediffusion repository was taken down. This mirror has the exact commit the WebUI needs.
%env STABLE_DIFFUSION_REPO=https://github.com/w-e-w/stablediffusion.git
# [FIX] Runs with the Python 3.10 environment, and requirements_versions.txt (pinned) instead of requirements.txt (unpinned).
# [SECURITY] --disable-safe-unpickle and --enable-insecure-extension are no longer on by default. The gradio link is public:
#   anyone who has it can use your WebUI. Add --gradio-auth user:password to require a login.
#   Only add --disable-safe-unpickle back if a .ckpt/.pt file you trust is refused as "may be malicious".
!COMMANDLINE_ARGS="--share --no-half-vae --xformers --gradio-queue" REQS_FILE="requirements_versions.txt" /content/sdenv/bin/python launch.py
# Use this command below to use cloudflare tunnel
#!COMMANDLINE_ARGS="--no-half-vae --xformers --gradio-queue --cloudflared" REQS_FILE="requirements_versions.txt" /content/sdenv/bin/python launch.py

## (OPTIONAL) LoRAs

### 1. After the gradio link show up, stop the first cell & clear the code output👆

### 2. Load the LoRA & launch the web ui

In [ ]:
# Copy the LoRA code from other LoRA setup (download the setup file after editing the LoRA code cell to avoid repeat input for next session)
# How-to download the setup file: Click 'File' menu -> 'Download' -> 'Download .ipynb'
# Load LoRA from Google Drive: https://youtu.be/G1QZfAPUMaM

# Detail Tweaker
#!curl -Lo /content/microsoftexcel/models/Lora/detailtweaker.safetensors https://huggingface.co/nolanaatama/dtltwkr/resolve/main/dtltwkr.safetensors

# Add More Details
#!curl -Lo /content/microsoftexcel/models/Lora/addmoredetails.safetensors https://huggingface.co/nolanaatama/ddmrdtls/resolve/main/ddmrdtls.safetensors

# Additional LoRA 1
#!curl ...

# Additional LoRA 2
#!curl ...

# Additional LoRA 3
#!curl ...

# ...

# Web UI tunnel
# [FIX] GitHub's Stability-AI/stablediffusion repository was taken down. This mirror has the exact commit the WebUI needs.
%env STABLE_DIFFUSION_REPO=https://github.com/w-e-w/stablediffusion.git
# [FIX] Runs with the Python 3.10 environment, and requirements_versions.txt (pinned) instead of requirements.txt (unpinned).
# [SECURITY] --disable-safe-unpickle and --enable-insecure-extension are no longer on by default. The gradio link is public:
#   anyone who has it can use your WebUI. Add --gradio-auth user:password to require a login.
#   Only add --disable-safe-unpickle back if a .ckpt/.pt file you trust is refused as "may be malicious".
!COMMANDLINE_ARGS="--share --no-half-vae --xformers --gradio-queue" REQS_FILE="requirements_versions.txt" /content/sdenv/bin/python launch.py
# Use this command below to use cloudflare tunnel
#!COMMANDLINE_ARGS="--no-half-vae --xformers --gradio-queue --cloudflared" REQS_FILE="requirements_versions.txt" /content/sdenv/bin/python launch.py

# 📚 GitHub for more: [_@nolanaatama_](https://github.com/nolanaatama)
# 📦 Repo: [Github](https://github.com/nolanaatama/sd-1click-colab)

Repaired for current Colab (Python 3.10 environment, pinned versions). Every change is marked `# [FIX]` or `# [SECURITY]`, and the details are in `epicrealism_REPAIR_REPORT.md`. The untouched original is `epicrealism_original_backup.ipynb`.